In [0]:
%sql
create connection if not exists earthquake_conn
type HTTP
options(
     host="https://earthquake.usgs.gov",
     port=443,
     base_path='/earthquakes/feed/v1.0/',
     bearer_token="na"
)

In [0]:
%py
from databricks.sdk import WorkspaceClient
w=WorkspaceClient()

conn=w.connections.get("earthquake_conn")
base_url=f"{conn.options['host']}{conn.options['base_path']}"

In [0]:
dll=https://earthquake.usgs.gov/earthquakes/feed/v1.1/summary/all_day.geojson

In [0]:
dbutils.widgets.text("catalog_name",'geek_dev')
catalog_name=dbutils.widgets.get("catalog_name")
catalog_name


In [0]:
%py
spark.sql(f"use catalog {catalog_name}")
spark.sql(f"use schema bronze")
spark.sql("create volume if not exists earthquake_data")

In [0]:
import requests
import json
import datetime
url=f'{base_url}summary/all_day.geojson'.replace('feed/v1.1','feed/v1.0')

response=requests.get(url)
if response.status_code != 200:
  raise Exception(f"Error: {response.status_code} - {response.text}")
data =response.json()
current_date=datetime.datetime.now().strftime("%Y-%m-%d")
dbutils.fs.put(f"/Volumes/geek_dev/bronze/earthquake_data/earthquake_data_{current_date}.json",json.dumps(data),overwrite=True)


